# Crawl: Getting Started

This notebook follows Kaggle's Maze Crawler starter notebook. It builds two simple agents for the `crawl` environment:

1. `agent_v1`: the factory moves north with a bug-move policy and jumps over walls when it can.
2. `agent_v2`: the factory keeps one scout alive. The scout uses a small snail-move memory to chase visible crystals and transfer energy back to the factory.


## Setup


In [ ]:
%%capture
!pip install -q 'kaggle-environments>=1.29.0'


In [ ]:
from kaggle_environments import make

env = make("crawl", configuration={"randomSeed": 42}, debug=True)
print("Environment ready. Width:", env.configuration.width)


## Helpers

The wall layout is a flat array of bitfields, one per cell. The direction bits are `NORTH = 1`, `EAST = 2`, `SOUTH = 4`, and `WEST = 8`. These helpers read walls, check open roads, find neighboring cells, and filter our robots.


In [ ]:
DIRS = {"NORTH": (0, 1), "SOUTH": (0, -1), "EAST": (1, 0), "WEST": (-1, 0)}
WALL_BIT = {"NORTH": 1, "EAST": 2, "SOUTH": 4, "WEST": 8}
OPPOSITE = {"NORTH": "SOUTH", "SOUTH": "NORTH", "EAST": "WEST", "WEST": "EAST"}


def wall_at(obs, config, col, row):
    """Wall bitfield at (col, row). Returns 0 if the cell is undiscovered."""
    idx = (row - obs.southBound) * config.width + col
    if 0 <= idx < len(obs.walls) and obs.walls[idx] != -1:
        return obs.walls[idx]
    return 0


def has_road(obs, config, col, row, direction):
    """True if there is no wall in `direction` from (col, row)."""
    return not (wall_at(obs, config, col, row) & WALL_BIT[direction])


def neighbor(col, row, direction):
    """Return the (col, row) one step in `direction`."""
    dc, dr = DIRS[direction]
    return col + dc, row + dr


def my_robots(obs):
    """Map of uid -> robot data for robots owned by us."""
    return {uid: data for uid, data in obs.robots.items() if data[4] == obs.player}


def my_factory(obs):
    """Return (uid, data) of our factory, or (None, None)."""
    for uid, data in my_robots(obs).items():
        if data[0] == 0:
            return uid, data
    return None, None


def parse_cell(key):
    """Parse a 'col,row' key into (col, row)."""
    c, r = key.split(",")
    return int(c), int(r)


## Bug Move

The factory needs to keep moving north as the maze scrolls south. If north is blocked, it uses `JUMP_NORTH` when available, then tries east or west as a sidestep.


In [ ]:
def factory_bug_north(uid, col, row, jump_cd, obs, config):
    """Bug move toward north: north > jump > east > west > idle."""
    if has_road(obs, config, col, row, "NORTH"):
        return "NORTH"
    if jump_cd == 0:
        return "JUMP_NORTH"
    if has_road(obs, config, col, row, "EAST"):
        return "EAST"
    if has_road(obs, config, col, row, "WEST"):
        return "WEST"
    return "IDLE"


def agent_v1(obs, config):
    """Factory only: bug-move north, jumping over walls when possible."""
    actions = {}
    for uid, data in my_robots(obs).items():
        rtype, col, row = data[0], data[1], data[2]
        jump_cd = data[6] if len(data) > 6 else 0
        if rtype == 0:
            actions[uid] = factory_bug_north(uid, col, row, jump_cd, obs, config)
    return actions


In [ ]:
env = make("crawl", configuration={"randomSeed": 42}, debug=True)
env.run([agent_v1, "random"])
for i, step in enumerate(env.steps[-1]):
    print(f"Player {i}: reward={step.reward}, status={step.status}")
env.render(mode="ipython", width=800, height=800)


## Snail Move

The scout keeps a short per-robot tabu list of the last two cells it visited. Each turn it chooses an open neighbor that gets closer to the target without stepping back into that list. The target resets the tabu list whenever it changes.


In [ ]:
# Per-scout snail-move state, persisted across turns.
_scout_state = {}  # uid -> {"target": (col, row), "tabu": [(col, row), ...]}


def snail_step(uid, col, row, target, obs, config):
    """Greedy step toward `target`, refusing to revisit the last 2 cells."""
    state = _scout_state.setdefault(uid, {"target": None, "tabu": []})
    if state["target"] != target:
        state["target"] = target
        state["tabu"] = []

    tc, tr = target
    candidates = []
    for direction in ("NORTH", "SOUTH", "EAST", "WEST"):
        if not has_road(obs, config, col, row, direction):
            continue
        nc, nr = neighbor(col, row, direction)
        if (nc, nr) in state["tabu"]:
            continue
        dist = abs(nc - tc) + abs(nr - tr)
        candidates.append((dist, direction))

    if not candidates:
        state["tabu"] = []
        return "IDLE"

    candidates.sort()
    action = candidates[0][1]
    state["tabu"] = (state["tabu"] + [(col, row)])[-2:]
    return action


def scout_action(uid, col, row, energy, factory_pos, obs, config):
    fc, fr = factory_pos
    if energy >= 75:
        for direction in ("NORTH", "SOUTH", "EAST", "WEST"):
            if neighbor(col, row, direction) == (fc, fr) and has_road(obs, config, col, row, direction):
                return f"TRANSFER_{direction}"
        return snail_step(uid, col, row, (fc, fr), obs, config)

    crystals = [parse_cell(key) for key in obs.crystals]
    if crystals:
        target = min(crystals, key=lambda cell: abs(cell[0] - col) + abs(cell[1] - row))
    else:
        target = (col, row + 5)
    return snail_step(uid, col, row, target, obs, config)


def agent_v2(obs, config):
    """Factory bug-moves north and keeps exactly one scout alive."""
    actions = {}
    robots = my_robots(obs)
    _, f_data = my_factory(obs)

    scout_count = sum(1 for data in robots.values() if data[0] == 1)

    for uid, data in robots.items():
        rtype, col, row, energy = data[0], data[1], data[2], data[3]
        jump_cd = data[6] if len(data) > 6 else 0
        build_cd = data[7] if len(data) > 7 else 0

        if rtype == 0:
            if scout_count == 0 and build_cd == 0 and energy >= config.scoutCost:
                actions[uid] = "BUILD_SCOUT"
            else:
                actions[uid] = factory_bug_north(uid, col, row, jump_cd, obs, config)
        elif rtype == 1 and f_data is not None:
            factory_pos = (f_data[1], f_data[2])
            actions[uid] = scout_action(uid, col, row, energy, factory_pos, obs, config)

    return actions


In [ ]:
_scout_state.clear()
env = make("crawl", configuration={"randomSeed": 42}, debug=True)
env.run([agent_v2, "random"])
for i, step in enumerate(env.steps[-1]):
    print(f"Player {i}: reward={step.reward}, status={step.status}")
env.render(mode="ipython", width=800, height=800)


## Save Your Submission

Kaggle expects an `agent` function in `main.py` at the root of the notebook working directory. The cell below writes the `agent_v2` strategy as `agent`.


In [ ]:
%%writefile main.py
DIRS = {"NORTH": (0, 1), "SOUTH": (0, -1), "EAST": (1, 0), "WEST": (-1, 0)}
WALL_BIT = {"NORTH": 1, "EAST": 2, "SOUTH": 4, "WEST": 8}


def wall_at(obs, config, col, row):
    idx = (row - obs.southBound) * config.width + col
    if 0 <= idx < len(obs.walls) and obs.walls[idx] != -1:
        return obs.walls[idx]
    return 0


def has_road(obs, config, col, row, direction):
    return not (wall_at(obs, config, col, row) & WALL_BIT[direction])


def neighbor(col, row, direction):
    dc, dr = DIRS[direction]
    return col + dc, row + dr


def my_robots(obs):
    return {uid: data for uid, data in obs.robots.items() if data[4] == obs.player}


def my_factory(obs):
    for uid, data in my_robots(obs).items():
        if data[0] == 0:
            return uid, data
    return None, None


def parse_cell(key):
    c, r = key.split(",")
    return int(c), int(r)


def factory_bug_north(uid, col, row, jump_cd, obs, config):
    if has_road(obs, config, col, row, "NORTH"):
        return "NORTH"
    if jump_cd == 0:
        return "JUMP_NORTH"
    if has_road(obs, config, col, row, "EAST"):
        return "EAST"
    if has_road(obs, config, col, row, "WEST"):
        return "WEST"
    return "IDLE"


_scout_state = {}


def snail_step(uid, col, row, target, obs, config):
    state = _scout_state.setdefault(uid, {"target": None, "tabu": []})
    if state["target"] != target:
        state["target"] = target
        state["tabu"] = []

    tc, tr = target
    candidates = []
    for direction in ("NORTH", "SOUTH", "EAST", "WEST"):
        if not has_road(obs, config, col, row, direction):
            continue
        nc, nr = neighbor(col, row, direction)
        if (nc, nr) in state["tabu"]:
            continue
        dist = abs(nc - tc) + abs(nr - tr)
        candidates.append((dist, direction))

    if not candidates:
        state["tabu"] = []
        return "IDLE"

    candidates.sort()
    action = candidates[0][1]
    state["tabu"] = (state["tabu"] + [(col, row)])[-2:]
    return action


def scout_action(uid, col, row, energy, factory_pos, obs, config):
    fc, fr = factory_pos
    if energy >= 75:
        for direction in ("NORTH", "SOUTH", "EAST", "WEST"):
            if neighbor(col, row, direction) == (fc, fr) and has_road(obs, config, col, row, direction):
                return f"TRANSFER_{direction}"
        return snail_step(uid, col, row, (fc, fr), obs, config)

    crystals = [parse_cell(key) for key in obs.crystals]
    if crystals:
        target = min(crystals, key=lambda cell: abs(cell[0] - col) + abs(cell[1] - row))
    else:
        target = (col, row + 5)
    return snail_step(uid, col, row, target, obs, config)


def agent(obs, config):
    actions = {}
    robots = my_robots(obs)
    _, f_data = my_factory(obs)

    scout_count = sum(1 for data in robots.values() if data[0] == 1)

    for uid, data in robots.items():
        rtype, col, row, energy = data[0], data[1], data[2], data[3]
        jump_cd = data[6] if len(data) > 6 else 0
        build_cd = data[7] if len(data) > 7 else 0

        if rtype == 0:
            if scout_count == 0 and build_cd == 0 and energy >= config.scoutCost:
                actions[uid] = "BUILD_SCOUT"
            else:
                actions[uid] = factory_bug_north(uid, col, row, jump_cd, obs, config)
        elif rtype == 1 and f_data is not None:
            factory_pos = (f_data[1], f_data[2])
            actions[uid] = scout_action(uid, col, row, energy, factory_pos, obs, config)

    return actions


## Submit to the Leaderboard

Run the write cell, save the Kaggle notebook, then use **Submit to Competition** in the right-hand Kaggle notebook panel. Kaggle will run `agent` from `main.py` against other submissions. See `AGENTS.md` in the competition data for the full CLI flow.


## Where to Go Next

- Build workers to remove walls in front of the factory with `REMOVE_NORTH`.
- Build miners and `TRANSFORM` them on mining nodes for steady income.
- Build multiple scouts and assign them to different crystals.
- Replace snail move with BFS or A* once you cache enough walls.
